# Good Words Attack on a Naive Bayes Spam Filter (White-Box)

**Author:** Teodora Demerdzhieva  
**Topic:** Adversarial Machine Learning - Evasion Attacks (Feature Obfuscation / "Good Words" Attacks)

This notebook demonstrates a good words attack (also known as a feature obfuscation attack) against a Naive Bayes spam filter, with full access to the trained model.

The filter is already trained and deployed. The attacker doesn't touch training data and doesn't retrain anything. A handful of ordinary words get appended to a spam message at the moment it's scored and nothing in the original message is removed or changed. The extra words are chosen because the filter associates them strongly with legitimate mail. A human reading the result still sees an obvious spam message. The filter's verdict flips anyway.

This is the reverse of a trojan attack. A trojan plants a trigger during training so the finished model misbehaves later. A good word attack needs no training access at all - it exploits how an already trained model scores text at prediction time. It's also the original context for the technique: Lowd and Meek published it in 2005 against exactly this kind of filter, under the name it still carries.

This is the white-box case: the model file is already in hand, so the strongest words get read straight off its internals, not guessed at.

## Attack Goal

| | White-Box Attack |
|---|---|
| Starting messages | Spam |
| Target verdict | Ham |
| Access to the model | Full (the trained classifier object) |
| Extra words allowed per message | Up to 30 |

## How It Works

A Naive Bayes classifier doesn't read a message, it tallies one. Every word it knows carries a fixed number of votes toward "spam" and a fixed number toward "ham", learned from training data. A word like "free" carries a heavy vote toward spam; a word like "lunch" barely moves the needle either way. At prediction time the filter sums every vote in the message and goes with whichever side wins. It never checks whether the votes came from the same sentence, contradict each other, or add up to a coherent message.

That's the vulnerability. Drop a handful of ordinary, ham-leaning words into the message, words that were never part of the original text, and the vote total moves without a single word of the actual spam changing. With the model file in hand, finding the strongest words is a matter of reading its own numbers back, not guessing.

## 1. Setup and Imports

In [1]:
import html
import random

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", None)
print("Setup complete.")

Setup complete.


## 2. A Small, Public Spam/Ham Dataset

This notebook runs on the SMS Spam Collection (Almeida, Gomez Hidalgo & Yamakami, 2011) - 5,574 real SMS messages, each tagged spam or ham, collected for spam-filtering research and reused in more text-classification tutorials than almost any other dataset this size. 

Six spam messages are held out of training and used as attack targets below.

In [2]:
try:
    pd.set_option("future.infer_string", False)
except Exception:
    pass  
try:
    pd.set_option("mode.string_storage", "python")
except Exception:
    pass

DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

sms = pd.read_csv(DATA_URL, sep="\t", header=None, names=["label", "text"], dtype=object)

# The raw corpus has HTML-escaped redaction placeholders (phone numbers and
# amounts were anonymized as e.g. &lt;#&gt; before release). Left as-is these
# tokenize into meaningless "words" like gt/lt/amp; unescaping first collapses
# them back to punctuation, which the vectorizer correctly ignores.
sms["text"] = sms["text"].apply(html.unescape)

TARGET_SPAM = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
    "U have won a nokia 6230 plus a free digital camera. This is what u get when u win our FREE auction. To take part send NOKIA to 83383 now. POBOX114/14TCR/W1 16",
    "Thanks for your ringtone order, reference number X29. Your mobile will be charged 4.50. Should your tone not arrive please call customer services 09065989180",
    "Want a new Video Phone? 750 anytime any network mins? Half price line rental free text for 3 months? Reply or call 08000930705 for free delivery",
    "We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p",
    "You have an important customer service announcement from PREMIER.",
]

sms = sms[~sms["text"].isin(TARGET_SPAM)]

spam_msgs = sms.loc[sms.label == "spam", "text"].tolist()
ham_msgs = sms.loc[sms.label == "ham", "text"].tolist()
random.shuffle(spam_msgs)
random.shuffle(ham_msgs)

print(f"Training pool: {len(spam_msgs)} spam + {len(ham_msgs)} ham messages")
print(f"{len(TARGET_SPAM)} example spam messages set aside as attack targets (not used in training)")

Training pool: 740 spam + 4825 ham messages
6 example spam messages set aside as attack targets (not used in training)


## 3. Train the Filter

A standard recipe: vectorize each message into word counts (the literal "bag"), then fit a Multinomial Naive Bayes model - for every word in the vocabulary, count how often it shows up in spam vs. ham messages during training, and turn that into a vote weight.

In [3]:
X = spam_msgs + ham_msgs
y = ["spam"] * len(spam_msgs) + ["ham"] * len(ham_msgs)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=1, stratify=y)

vec = CountVectorizer(stop_words="english")
clf = MultinomialNB(alpha=1.0).fit(vec.fit_transform(X_train), y_train)
accuracy = clf.score(vec.transform(X_test), y_test)

print(f"Filter trained: {accuracy:.1%} held-out accuracy, {len(vec.get_feature_names_out())} vocabulary words")

Filter trained: 98.4% held-out accuracy, 7715 vocabulary words


## 4. Look Inside the Model

With the real `MultinomialNB` object in hand, its vote-weight table reads out directly. For every vocabulary word, scikit-learn stores `log P(word | spam)` and `log P(word | ham)`. Subtract one from the other and each word gets a single score: how strongly it votes ham. The model hands over the entire table for free and no attack is needed yet, just inspection.

In [4]:
names = vec.get_feature_names_out()
classes = list(clf.classes_)
spam_row, ham_row = classes.index("spam"), classes.index("ham")

# log P(word | ham) - log P(word | spam): a positive number means the word
# votes ham; the larger the number, the stronger the vote.
llr_ham = clf.feature_log_prob_[ham_row] - clf.feature_log_prob_[spam_row]
ranking = sorted(zip(names, llr_ham), key=lambda t: -t[1])

print("Strongest HAM-voting words, read straight out of the model:")
for word, score in ranking[:15]:
    print(f"  {word:<15} vote strength: {score:.2f}")

Strongest HAM-voting words, read straight out of the model:
  lor             vote strength: 4.27
  da              vote strength: 4.23
  later           vote strength: 3.97
  come            vote strength: 3.84
  way             vote strength: 3.70
  ask             vote strength: 3.65
  doing           vote strength: 3.65
  said            vote strength: 3.65
  yeah            vote strength: 3.60
  sure            vote strength: 3.48
  morning         vote strength: 3.45
  lol             vote strength: 3.39
  gud             vote strength: 3.37
  cos             vote strength: 3.35
  feel            vote strength: 3.32


A couple of these (`lor`, `da`) are casual discourse particles, common among this dataset's texters, not standard English dictionary words. The model doesn't know that. It only knows these tokens showed up disproportionately in the ham half of its training data, which is exactly the point: it has no idea what anything means, only what correlates.

## 5. Craft the Attack

The recipe:
1. Sort candidate words by vote strength toward ham.
2. Append words to the message from that list, strongest first, one at a time, stopping the instant the model's own verdict flips.
3. Never touch, remove, or reorder the original message text. Never exceed the word budget.

In [5]:
def craft_attack(message, word_ranking, predict_fn, target_label, word_budget, confidence=0.55):
    '''
    Greedily append the strongest available words, one at a time, until the
    model's own prediction flips to `target_label` with at least `confidence`
    probability -- or we run out of ranked words or the word budget.
    The original message text is never changed, only added to.
    '''
    added = []
    before_label, before_prob = predict_fn(message)   # the untouched message's own verdict
    label, prob, candidate = before_label, before_prob, message
    for word, _ in word_ranking:
        if len(added) >= word_budget or (label == target_label and prob >= confidence):
            break
        added.append(word)
        candidate = message + " " + " ".join(added)
        label, prob = predict_fn(candidate)
    return {
        "message": message, "augmented": candidate, "added_words": " ".join(added),
        "n_added": len(added),
        "before_label": before_label, "before_prob": before_prob,
        "after_label": label, "after_prob": prob,
        "flipped": label == target_label,
    }


def predict(text):
    probs = clf.predict_proba(vec.transform([text]))[0]
    idx = probs.argmax()
    return clf.classes_[idx], probs[idx]


WORD_BUDGET = 30  
print("Attack function ready.")

Attack function ready.


## 6. Results

In [6]:
results = [craft_attack(m, ranking, predict, "ham", WORD_BUDGET) for m in TARGET_SPAM]

table = pd.DataFrame([{
    "Original message (truncated)": r["message"][:58] + ("…" if len(r["message"]) > 58 else ""),
    "Words added": r["added_words"] or "—",
    "# added": r["n_added"],
    "Before": f'{r["before_label"]} ({r["before_prob"]:.0%})',
    "After": f'{r["after_label"]} ({r["after_prob"]:.0%})',
    "Flipped?": "yes" if r["flipped"] else "NO",
} for r in results])

print(f"{sum(r['flipped'] for r in results)}/{len(results)} messages flipped to ham, "
      f"using an average of {np.mean([r['n_added'] for r in results]):.1f} added words "
      f"(budget was {WORD_BUDGET}).")
table

6/6 messages flipped to ham, using an average of 6.3 added words (budget was 30).


,Original message (truncated),Words added,# added,Before,After,Flipped?
0,Free entry in 2 a wkly comp to win FA Cup final tkts 21st …,lor da later come way ask doing said yeah sure,10,spam (100%),ham (84%),yes
1,U have won a nokia 6230 plus a free digital camera. This i…,lor da later come way ask doing said,8,spam (100%),ham (95%),yes
2,"Thanks for your ringtone order, reference number X29. Your…",lor da later come way ask doing,7,spam (100%),ham (98%),yes
3,Want a new Video Phone? 750 anytime any network mins? Half…,lor da later come way ask doing said yeah,9,spam (100%),ham (98%),yes
4,We know someone who you know that fancies you. Call 090580…,lor da,2,spam (100%),ham (88%),yes
5,You have an important customer service announcement from P…,lor da,2,spam (100%),ham (67%),yes


In [7]:
example = results[0]
print("=== Example transformation ===\n")
print("BEFORE:", example["message"])
print("\nAFTER: ", example["augmented"])
print(f"\nVerdict flipped from {example['before_label']} ({example['before_prob']:.0%}) to "
      f"{example['after_label']} ({example['after_prob']:.0%}) after adding just "
      f"{example['n_added']} word(s): \"{example['added_words']}\".")
print("\nThe message is still, obviously, spam. The model disagrees.")

=== Example transformation ===

BEFORE: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

AFTER:  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's lor da later come way ask doing said yeah sure

Verdict flipped from spam (100%) to ham (84%) after adding just 10 word(s): "lor da later come way ask doing said yeah sure".

The message is still, obviously, spam. The model disagrees.


## Why This Works

Naive Bayes assumes every word's contribution to the verdict is independent of every other word. That's what makes the model fast and cheap to train, and it's what lets an attacker add votes without engaging with what the message actually says. Any classifier that scores text by summing independent per-word contributions inherits this - bag-of-words logistic regression has the identical exploitable structure, not just Naive Bayes.


## Defenses

- Don't expose the trained model file for public download without a genuine need to (MITRE ATLAS AML.M0005, Control Access to ML Models and Data at Rest).
- Move beyond pure bag-of-words - n-gram features at minimum, contextual encoders ideally, since neither shares the "sum of independent per-word votes" structure this attack relies on.
- Adversarial training - include good-word-attacked examples in the training data so the model learns to discount bolted-on ham-leaning padding.
- Ensemble with a secondary check - a second model scoring coherence, or flagging word-salad padding, makes a single flipped vote much harder to land (AML.M0006, Use Ensemble Methods).

## References

- Lowd, D. & Meek, C. (2005) - [Good Word Attacks on Statistical Spam Filters](https://personal.utdallas.edu/~mxk055100/courses/dmsec_files/125.pdf), CEAS 2005
- Almeida, T.A., Gomez Hidalgo, J.M. & Yamakami, A. (2011) - [Contributions to the Study of SMS Spam Filtering: New Collection and Results](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), DOCENG'11, source of the SMS Spam Collection used throughout
- MITRE ATLAS - [AML.T0043.000, White-Box Optimization](https://d3fend.mitre.org/offensive-technique/attack/AML.T0043.000/) (parent technique [AML.T0043, Craft Adversarial Data](https://d3fend.mitre.org/offensive-technique/attack/AML.T0043/))
- MITRE ATLAS - [AML.T0015, Evade AI Model](https://d3fend.mitre.org/offensive-technique/attack/AML.T0015/)
- MITRE ATLAS - [mitigations catalog](https://atlas.mitre.org/mitigations/), source for AML.M0005 and AML.M0006 cited in the defense section
- OWASP ML Security Top 10 - [ML01:2023, Input Manipulation Attack](https://owasp.org/www-project-machine-learning-security-top-10/docs/ML01_2023-Input_Manipulation_Attack)
